# Predictive workspace: Laya fine-tuning pilot

Run on a **GPU runtime** in Google Colab (Runtime → Change runtime type). Start with a single T4, when available. This notebook follows the upstream Laya RLCD choice-task loop and invokes the repository's tested package. It does not use a generative chat fine-tuning format.

**No GPU run has been verified by the author.** The CPU random-fixture pipeline passes; that is compatibility evidence, not model quality. The data consists of 282 AI-authored synthetic examples. Inspect the label policy and dataset card before spending compute.

Full training is disabled by default. A one-step GPU probe records memory/time. No API credentials, Drive mount, paid generation, model publication, or Hub uploads are used. Save the artifact zip before the session expires.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys, time

REPO_URL = "https://github.com/ivankqw/predictive-chat-ui.git"
REPO_BRANCH = "codex/laya-finetune-preparation"
# Immutable tested package revision; later notebook/docs commits do not change this code.
PACKAGE_COMMIT = "b9120e8fb30c21b115f32da2cb3cf6de1a8eaf93"
CHECKOUT = Path("/content/predictive-chat-ui")
RUN_PROBE = True
RUN_TRAINING = False  # Explicitly change to True after inspecting the probe.
RUN_ID = time.strftime("%Y%m%d-%H%M%S", time.gmtime())

def run(command, cwd=None):
    subprocess.run([str(x) for x in command], cwd=cwd, check=True)

# Keep the Colab kernel intact; run model code in a pinned Python environment.
run([sys.executable, "-m", "pip", "install", "uv==0.12.19"])
VENV = Path("/content/laya-python312")
PYTHON = VENV / "bin/python"
if not PYTHON.exists():
    run([sys.executable, "-m", "uv", "venv", "--python", "3.12.13", "--seed", VENV])
run([PYTHON, "-c", "import sys; assert sys.version_info[:3] == (3, 12, 13), sys.version; print(sys.version)"])

if not CHECKOUT.exists():
    run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, CHECKOUT])
if subprocess.check_output(["git", "status", "--porcelain"], cwd=CHECKOUT, text=True).strip():
    raise RuntimeError("Checkout has edits; preserve them and use a fresh Colab checkout.")
run(["git", "fetch", "origin", REPO_BRANCH], cwd=CHECKOUT)
run(["git", "checkout", "--detach", PACKAGE_COMMIT], cwd=CHECKOUT)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CHECKOUT, text=True).strip() == PACKAGE_COMMIT
OUTPUT = CHECKOUT / "finetune/outputs" / RUN_ID
OUTPUT.mkdir(parents=True, exist_ok=False)
os.environ["HF_HOME"] = "/content/laya-hf-cache"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
print("Pinned package", PACKAGE_COMMIT, "artifacts", OUTPUT)


## Install and inspect hardware

The notebook supports a Python 3.13 Colab kernel by installing a separate Python 3.12.13 environment with pinned uv. No runtime downgrade or kernel restart is needed. All PyTorch/model operations run in fresh subprocesses so a Colab kernel's preloaded packages cannot silently select an old runtime. CUDA 12.4 wheels require a compatible NVIDIA driver. If installation or the hardware check fails, stop and choose a supported GPU runtime; do not disable TLS or checksum verification.


In [ ]:
run([PYTHON, "-m", "pip", "install", "torch==2.6.0", "--index-url", "https://download.pytorch.org/whl/cu124"])
run([PYTHON, "-m", "pip", "install", "-r", CHECKOUT / "finetune/requirements.txt"])
run([PYTHON, "-c", "import torch; assert torch.cuda.is_available(), 'Select a Colab GPU runtime'; print(torch.__version__, torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory)"])
(OUTPUT / "packages.txt").write_text(subprocess.check_output([PYTHON, "-m", "pip", "freeze"], text=True))

def pipeline(*args):
    run([PYTHON, "-m", "finetune.runtime", *args], cwd=CHECKOUT)

run([PYTHON, "-m", "finetune.data_tools", "verify"], cwd=CHECKOUT)
run([PYTHON, "-m", "pytest", "-q", "finetune/tests"], cwd=CHECKOUT)
run([PYTHON, "-m", "finetune.smoke", "--output", OUTPUT / "offline-smoke"], cwd=CHECKOUT)


## Fetch pinned checkpoint and measure one GPU optimizer step

The public checkpoint needs no Hugging Face token. The package verifies its pinned revision and expected weight SHA256 before use. Never replace the expected hash to make a download pass. Reuse verified base files, but do not train from probe weights.

The probe uses batch 1 and one accumulation step. This exercises forward, backward and Adam state allocation. Full training also uses batch 1. If it OOMs, a larger GPU is a better next experiment than assuming multiple T4s pool memory.


In [ ]:
BASE = CHECKOUT / "finetune/outputs/base"
if not BASE.exists():
    pipeline("fetch-base", "--output", BASE)
else:
    run([PYTHON, "-c", "from pathlib import Path; from finetune.data_tools import digest; from finetune.runtime import BASE_SHA256; import sys; assert digest(Path(sys.argv[1])/'model.safetensors') == BASE_SHA256", BASE], cwd=CHECKOUT)

if RUN_PROBE:
    pipeline("train", "--model", BASE, "--device", "cuda", "--epochs", "1", "--max-steps", "1",
             "--batch-size", "1", "--accumulation", "1", "--output", OUTPUT / "probe")
    print((OUTPUT / "probe/training.json").read_text())
else:
    print("GPU memory probe skipped; resource needs remain unknown.")


## Full training, separate calibration and gate selection

Change `RUN_TRAINING=True` only when ready. Pilot defaults: four epochs, batch 1, accumulation 8. The 141 train rows yield 18 optimizer updates per epoch. `checkpoint_latest` is saved after each epoch; it is an inference checkpoint, not exact optimizer resume.

Only train rows enter optimization. Calibration rows fit serving-compatible choice temperature. Validation selects the gate. Test reports are generated together after those steps and must not be used to retune the model or gate.


In [ ]:
if RUN_TRAINING:
    pipeline("evaluate", "--model", BASE, "--device", "cuda", "--split", "validation", "--select-gate",
             "--output", OUTPUT / "base-validation")
    pipeline("train", "--model", BASE, "--device", "cuda", "--batch-size", "1", "--accumulation", "8",
             "--output", OUTPUT / "run")
    pipeline("calibrate", "--model", OUTPUT / "run/checkpoint_latest", "--device", "cuda",
             "--output", OUTPUT / "calibrated")
    pipeline("evaluate", "--model", OUTPUT / "calibrated", "--device", "cuda", "--split", "validation",
             "--select-gate", "--output", OUTPUT / "adapted-validation")
else:
    print("Full training and model-quality evaluation are off. Inspect the probe and set RUN_TRAINING=True to proceed.")


In [ ]:
if RUN_TRAINING:
    pipeline("evaluate", "--model", BASE, "--device", "cuda", "--split", "test",
             "--gate", OUTPUT / "base-validation/gate.json", "--output", OUTPUT / "base-test")
    pipeline("evaluate", "--model", OUTPUT / "calibrated", "--device", "cuda", "--split", "test",
             "--gate", OUTPUT / "adapted-validation/gate.json", "--output", OUTPUT / "adapted-test")
    run([PYTHON, "-m", "finetune.compare", "--before", OUTPUT / "base-test/report.json",
         "--after", OUTPUT / "adapted-test/report.json", "--output", OUTPUT / "comparison.json"], cwd=CHECKOUT)
    comparison = json.loads((OUTPUT / "comparison.json").read_text())
    print(json.dumps(comparison["deltas"], indent=2))
    print("Read support counts, false suggestions, coverage, risk and all slices before recommending this checkpoint.")


## Save artifacts and decide whether more compute is needed

A successful one-step probe gives measured peak memory on **your allocated GPU**. If batch 1 still runs out of memory, try a 24 GB L4 or a 40 GB A100. Upstream's two-T4 configuration is an alternative for throughput; DDP replicates parameters and optimizer state and does not combine GPU memory. No specific Colab GPU or duration is guaranteed.

Report real baseline/adapted accuracy, per-label recall, false suggestions on `none`, negatives and topic questions, coverage/risk, Brier/NLL/ECE and warm latency on matching hardware. Synthetic pilot results are not evidence of real-user reliability. Ask for an independent label audit before expanding training. Keep the current production model until the results justify an explicit opt-in deployment.

The archive contains the run artifacts, not the base checkpoint or Hub cache. It may be large because it includes the adapted model. Colab's file download action is user-initiated; the zip is not uploaded to a public model hub.


In [ ]:
import shutil
ARCHIVE = Path(shutil.make_archive(str(OUTPUT) + "-artifacts", "zip", root_dir=OUTPUT))
print("Saved", ARCHIVE, "bytes", ARCHIVE.stat().st_size)
try:
    from google.colab import files
    files.download(str(ARCHIVE))
except ImportError:
    print("Outside Colab: copy the archive from the path above.")
